In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
print(os.getcwd())
print(os.getcwd())
print(os.path.exists("datos/credit_risk_dataset.csv"))


/workspaces/Proyecto-Credit-Risk-Dataset/cuadernos
/workspaces/Proyecto-Credit-Risk-Dataset/cuadernos
False


In [13]:
df = pd.read_csv(os.path.join("..", "datos", "credit_risk_dataset.csv"))

In [14]:
# %% [markdown]
# ## 2. Manejo de datos faltantes  (Integrante B)
faltantes = pd.DataFrame({
    "faltantes": df.isna().sum(),
    "porcentaje": (df.isna().mean() * 100).round(2),
})
faltantes = faltantes[faltantes["faltantes"] > 0]
print(faltantes)
 
cols_nulos = faltantes.index.tolist()
print("\nFilas con al menos un faltante:", df.isna().any(axis=1).sum())
print("Filas con ambos faltantes:", df[cols_nulos].isna().all(axis=1).sum())

                   faltantes  porcentaje
person_emp_length        895        2.75
loan_int_rate           3116        9.56

Filas con al menos un faltante: 3943
Filas con ambos faltantes: 68


In [25]:
edades_raras = (df["person_age"] > 100).sum()
antig_rara = (df["person_emp_length"] > 60).sum()
print(f"Personas con edad mayor a 100 años: {edades_raras}")
print(f"Personas con más de 60 años de antigüedad laboral: {antig_rara}")
 
df2 = df.copy()
df2.loc[df2["person_age"] > 100, "person_age"] = np.nan
df2.loc[df2["person_emp_length"] > 60, "person_emp_length"] = np.nan
print("\nEstos valores imposibles se convirtieron en datos faltantes (NaN).")

Personas con edad mayor a 100 años: 5
Personas con más de 60 años de antigüedad laboral: 2

Estos valores imposibles se convirtieron en datos faltantes (NaN).


In [26]:
print("Costo de ELIMINAR filas con datos faltantes:")
for col in df2.columns[df2.isna().any()]:
    cantidad = df2[col].isna().sum()
    porcentaje = df2[col].isna().mean() * 100
    print(f"  Si elimino los nulos de {col}: pierdo {cantidad} filas ({porcentaje:.2f}%)")
 
total = df2.isna().any(axis=1).sum()
print(f"  Si elimino cualquier fila con algún nulo: pierdo {total} filas "
      f"({total / len(df2) * 100:.2f}%)")
 
print('La desicion final es imputar, ya que eliminar filas con datos faltantes puede llevar a perder una gran cantidad de información valiosa, especialmente si los datos faltantes no son aleatorios. Imputar los valores faltantes permite mantener la integridad del conjunto de datos y aprovechar al máximo la información disponible.')

Costo de ELIMINAR filas con datos faltantes:
  Si elimino los nulos de person_age: pierdo 5 filas (0.02%)
  Si elimino los nulos de person_emp_length: pierdo 897 filas (2.75%)
  Si elimino los nulos de loan_int_rate: pierdo 3116 filas (9.56%)
  Si elimino cualquier fila con algún nulo: pierdo 3949 filas (12.12%)
La desicion final es imputar, ya que eliminar filas con datos faltantes puede llevar a perder una gran cantidad de información valiosa, especialmente si los datos faltantes no son aleatorios. Imputar los valores faltantes permite mantener la integridad del conjunto de datos y aprovechar al máximo la información disponible.


In [29]:
df_imp = df2.copy()
 
df_imp["loan_int_rate"] = df_imp["loan_int_rate"].fillna(
    df_imp.groupby("loan_grade")["loan_int_rate"].transform("median"))
df_imp["person_emp_length"] = df_imp["person_emp_length"].fillna(
    df_imp.groupby("person_home_ownership")["person_emp_length"].transform("median"))
df_imp["person_age"] = df_imp["person_age"].fillna(df_imp["person_age"].median())
 
# %% 7. Resultado después de imputar
print("Datos faltantes por columna después de imputar:")
print(df_imp.isna().sum())
print("\nTotal de datos faltantes:", df_imp.isna().sum().sum())
 
print("\nPrimeras 5 filas del dataframe imputado:")
print(df_imp.head())
 
# Filas que originalmente tenían un dato faltante, ya imputadas
filas_imputadas = df.index[df["loan_int_rate"].isna() | df["person_emp_length"].isna()]
comparacion = pd.DataFrame({
    "tasa_original": df.loc[filas_imputadas, "loan_int_rate"],
    "tasa_imputada": df_imp.loc[filas_imputadas, "loan_int_rate"],
    "antig_original": df.loc[filas_imputadas, "person_emp_length"],
    "antig_imputada": df_imp.loc[filas_imputadas, "person_emp_length"],
    "calificacion": df_imp.loc[filas_imputadas, "loan_grade"],
    "vivienda": df_imp.loc[filas_imputadas, "person_home_ownership"],
})

 
print("\nDimensiones del dataframe imputado:", df_imp.shape)

Datos faltantes por columna después de imputar:
person_age                    0
person_income                 0
person_home_ownership         0
person_emp_length             0
loan_intent                   0
loan_grade                    0
loan_amnt                     0
loan_int_rate                 0
loan_status                   0
loan_percent_income           0
cb_person_default_on_file     0
cb_person_cred_hist_length    0
dtype: int64

Total de datos faltantes: 0

Primeras 5 filas del dataframe imputado:
   person_age  person_income person_home_ownership  person_emp_length  \
0        22.0          59000                  RENT                3.0   
1        21.0           9600                   OWN                5.0   
2        25.0           9600              MORTGAGE                1.0   
3        23.0          65500                  RENT                4.0   
4        24.0          54400                  RENT                8.0   

  loan_intent loan_grade  loan_amnt  loan_int